In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

# ========= USER CONFIG =========
TICKERS = ["SMCI", "TQQQ", "SOXL", "COIN", "MSTR", "HOOD", "PLTR", "ASML", "TSM", "AMZN", "CRCL"]          # 可改成多个，例如 ["SMCI", "MSTR", "HOOD"]
START_DATE = "2024-01-01"   # 起始日期
END_DATE = None             # None = 到今天
DATA_DIR = "reversal_data"  # CSV输出目录
# ==============================


def download_reversal_data(ticker: str, start_date: str, end_date=None) -> pd.DataFrame:
    """
    下载 Yahoo Finance 历史数据并整理成：
    Date, Open, High, Low, Adj Close, Max Drop

    Max Drop = (前一交易日 Adj Close - 当日 Low) / 前一交易日 Adj Close
    """
    df = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False,
        threads=False,
    )

    if df.empty:
        print(f"{ticker}: no data downloaded.")
        return pd.DataFrame()

    # 某些 yfinance 版本会返回 MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    required_cols = ["Open", "High", "Low", "Adj Close"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{ticker}: missing required columns: {missing}")

    df = df[required_cols].copy()
    df = df.reset_index()

    # 标准化 Date
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # 先升序，方便用 shift(1) 取前一交易日 Adj Close
    df = df.sort_values("Date").reset_index(drop=True)

    # 数值清洗
    for col in ["Open", "High", "Low", "Adj Close"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 前一交易日 Adj Close
    prev_adj_close = df["Adj Close"].shift(1)

    # Max Drop = (Prev Adj Close - Today Low) / Prev Adj Close
    df["Max Drop"] = (prev_adj_close - df["Low"]) / prev_adj_close

    # 第一行没有前一日，删掉
    df = df.dropna(subset=["Date", "Open", "High", "Low", "Adj Close", "Max Drop"]).copy()

    # 最终列顺序
    df = df[["Date", "Open", "High", "Low", "Adj Close", "Max Drop"]]

    # 为了与你之前 Excel 使用习惯一致：最新日期在最上面
    df = df.sort_values("Date", ascending=False).reset_index(drop=True)

    return df


def save_ticker_csv(df: pd.DataFrame, ticker: str, data_dir: str) -> str:
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    file_path = os.path.join(data_dir, f"{ticker}.csv")
    df.to_csv(file_path, index=False)
    return file_path


def main():
    print("Updating reversal CSV datasets...\n")

    for ticker in TICKERS:
        try:
            df = download_reversal_data(ticker, START_DATE, END_DATE)
            if df.empty:
                print(f"{ticker}: skipped (empty data)\n")
                continue

            out_path = save_ticker_csv(df, ticker, DATA_DIR)
            print(f"{ticker}: saved {len(df)} rows -> {out_path}")

        except Exception as e:
            print(f"{ticker}: failed -> {e}")

    print("\nDone.")


if __name__ == "__main__":
    main()

Updating reversal CSV datasets...

SMCI: saved 549 rows -> reversal_data/SMCI.csv
TQQQ: saved 549 rows -> reversal_data/TQQQ.csv
SOXL: saved 549 rows -> reversal_data/SOXL.csv
COIN: saved 549 rows -> reversal_data/COIN.csv
MSTR: saved 549 rows -> reversal_data/MSTR.csv
HOOD: saved 549 rows -> reversal_data/HOOD.csv
PLTR: saved 549 rows -> reversal_data/PLTR.csv
ASML: saved 549 rows -> reversal_data/ASML.csv
TSM: saved 549 rows -> reversal_data/TSM.csv
AMZN: saved 549 rows -> reversal_data/AMZN.csv
CRCL: saved 192 rows -> reversal_data/CRCL.csv

Done.
